In [1]:
import torch
import numpy as np

from src.nano_maker.skeleton import Skeleton
from src.nano_maker.container import configs

from src.nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

In [2]:
c = configs.skeleton_config
model = Skeleton(n_embd=c['n_embd'], n_head=c['n_head'], n_layers=c['n_layers'],
                      block_size=c['block_size'], map4_res=c['map4_res'], max_angstroms=c['max_angstroms'],dropout=c['dropout'])

In [3]:
from src.paths import *
sk_wt_path = PROJECT_ROOT / "src/nano_maker/container/skeleton_e3.pt"

In [4]:
prototype_weights = torch.load(sk_wt_path, map_location="cpu")

In [5]:
print(type(prototype_weights))

<class 'dict'>


In [6]:
model.load_state_dict(prototype_weights["model_state_dict"])

<All keys matched successfully>

In [8]:
device = 'cpu'
block_size = c["block_size"]
max_angstroms = c["max_angstroms"]

In [9]:
sample_smiles = "CCNC(=O)[C@H]1CCCN1C(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)[C@H](CO)NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)CCc1ccc(Cl)cc1 |wU:57.63,12.20,wD:45.57,31.44,23.28,5.4,63.67,(30.68,-5.02,;30.68,-6.54,;29.36,-7.34,;28.03,-6.58,;26.73,-7.34,;28,-5.06,;29.2,-4.18,;28.69,-2.75,;27.18,-2.75,;26.75,-4.2,;25.41,-4.94,;25.41,-6.48,;24.09,-4.18,;24.09,-2.64,;25.44,-1.88,;25.44,-.33,;26.79,.42,;26.79,1.95,;28.13,2.73,;25.47,2.74,;22.76,-4.94,;21.41,-4.16,;21.44,-2.62,;20.08,-4.9,;20.06,-6.45,;21.4,-7.24,;21.38,-8.77,;22.73,-6.47,;18.74,-4.13,;17.42,-4.89,;17.41,-6.42,;16.1,-4.12,;16.1,-2.59,;15.64,-1.13,;14.17,-.64,;14.17,.9,;15.65,1.37,;16.26,2.77,;17.81,2.93,;18.71,1.7,;18.07,.28,;16.54,.12,;14.77,-4.87,;13.43,-4.1,;13.43,-2.55,;12.08,-4.86,;12.08,-6.38,;13.4,-7.18,;13.4,-8.72,;14.72,-9.49,;16.07,-8.73,;17.39,-9.51,;16.07,-7.19,;14.75,-6.42,;10.76,-4.07,;9.44,-4.83,;9.41,-6.38,;8.11,-4.06,;8.12,-2.52,;9.44,-1.77,;6.77,-4.81,;5.44,-4.03,;5.45,-2.51,;4.1,-4.8,;4.09,-6.34,;5.42,-7.12,;6.83,-6.5,;7.85,-7.64,;7.06,-8.98,;7.53,-10.44,;6.5,-11.57,;5,-11.24,;4.52,-9.78,;5.57,-8.66,;2.78,-4.03,;1.43,-4.77,;1.45,-6.31,;.08,-4.03,;-1.37,-4.99,;-2.89,-4.2,;-2.98,-2.49,;-4.53,-1.72,;-5.95,-2.65,;-7.46,-1.8,;-5.86,-4.38,;-4.33,-5.15,)|"

sample_map4 = torch.tensor(smiles_fingerprint(sample_smiles), dtype=torch.float32).unsqueeze(0)
sample_skeleton = model.generate(sample_map4)
sample_skeleton

tensor([[ 1.6207e+01,  2.0069e+00,  1.3893e+00],
        [ 1.5535e+01,  1.8340e+00,  1.4582e+00],
        [ 1.4690e+01,  2.4417e+00,  1.3962e+00],
        [ 1.4064e+01,  3.1000e-01,  1.6092e+00],
        [ 1.3815e+01,  9.5767e-01,  1.8016e+00],
        [ 1.3545e+01, -2.8970e-01,  2.0288e+00],
        [ 1.3748e+01,  1.6801e+00,  8.6265e-01],
        [ 1.2976e+01,  3.2267e-01,  1.8442e+00],
        [ 1.2980e+01,  9.4945e-02,  1.3104e+00],
        [ 1.2861e+01,  5.6289e-01,  1.3290e+00],
        [ 1.2629e+01,  2.1730e+00,  1.5148e+00],
        [ 1.2575e+01,  6.2654e-01,  1.3666e+00],
        [ 1.1943e+01,  2.5520e+00,  2.4304e+00],
        [ 1.1928e+01,  1.9978e-01,  1.8564e+00],
        [ 1.1644e+01,  2.7692e+00,  1.1338e+00],
        [ 1.1408e+01,  2.4485e+00,  1.6228e+00],
        [ 1.1235e+01,  1.9196e+00,  1.4689e+00],
        [ 1.1135e+01,  5.4602e-01,  2.4271e+00],
        [ 1.0790e+01,  7.4617e-01,  1.7570e+00],
        [ 1.0513e+01,  1.2257e+00,  1.6308e+00],
        [ 1.0345e+01

In [11]:
# Pocket Skeleton Generation algorithm prototyping

In [13]:
from src.nano_maker.modules.nAAno.radialseeker import RadialSeeker

In [14]:
r_mod = RadialSeeker(100, 33, False)

In [15]:
def process_skeleton(pocket_skeleton):
    output = []
    for vector in pocket_skeleton:
        output.append(vector.detach().flatten().tolist())
    return output

In [16]:
processed_sample = process_skeleton(sample_skeleton)
processed_sample

[[16.207429885864258, 2.0069310665130615, 1.3892793655395508],
 [15.535057067871094, 1.8340339660644531, 1.4582388401031494],
 [14.690017700195312, 2.4416568279266357, 1.3962090015411377],
 [14.063724517822266, 0.309998482465744, 1.6092350482940674],
 [13.814943313598633, 0.9576690793037415, 1.8015726804733276],
 [13.544590950012207, -0.2896958887577057, 2.028754711151123],
 [13.748334884643555, 1.6801369190216064, 0.8626493215560913],
 [12.976126670837402, 0.3226749300956726, 1.8442462682724],
 [12.980278015136719, 0.09494516253471375, 1.3104428052902222],
 [12.861181259155273, 0.562890887260437, 1.3289974927902222],
 [12.628862380981445, 2.173015594482422, 1.5148016214370728],
 [12.575322151184082, 0.6265423893928528, 1.3666160106658936],
 [11.942704200744629, 2.5520288944244385, 2.4304473400115967],
 [11.928483009338379, 0.1997820883989334, 1.8563735485076904],
 [11.643533706665039, 2.769176483154297, 1.1338350772857666],
 [11.40845775604248, 2.448460817337036, 1.622824788093567],
 

In [23]:
true_sample = []
for vector in processed_sample:
    appending = True
    if abs(vector[0]) < 0.33:
        break
    true_sample.append(vector)

true_sample

[[16.207429885864258, 2.0069310665130615, 1.3892793655395508],
 [15.535057067871094, 1.8340339660644531, 1.4582388401031494],
 [14.690017700195312, 2.4416568279266357, 1.3962090015411377],
 [14.063724517822266, 0.309998482465744, 1.6092350482940674],
 [13.814943313598633, 0.9576690793037415, 1.8015726804733276],
 [13.544590950012207, -0.2896958887577057, 2.028754711151123],
 [13.748334884643555, 1.6801369190216064, 0.8626493215560913],
 [12.976126670837402, 0.3226749300956726, 1.8442462682724],
 [12.980278015136719, 0.09494516253471375, 1.3104428052902222],
 [12.861181259155273, 0.562890887260437, 1.3289974927902222],
 [12.628862380981445, 2.173015594482422, 1.5148016214370728],
 [12.575322151184082, 0.6265423893928528, 1.3666160106658936],
 [11.942704200744629, 2.5520288944244385, 2.4304473400115967],
 [11.928483009338379, 0.1997820883989334, 1.8563735485076904],
 [11.643533706665039, 2.769176483154297, 1.1338350772857666],
 [11.40845775604248, 2.448460817337036, 1.622824788093567],
 

In [24]:
# convert everything into raw angstrom coordinates
angstrom_output = []
for vector in true_sample:
    angstrom_output.append(r_mod.radial_to_xyz(vector))
angstrom_output

[[-6.734169765180114, 14.448926127937195, 2.925794734487335],
 [-4.016766858354317, 14.904996348343234, 1.74489712118305],
 [-11.065344000905538, 9.318996844158944, 2.551681796821662],
 [13.383470121049884, 4.287071199307037, -0.5404584759555853],
 [7.738755359836917, 10.99905055727996, -3.159938460941669],
 [11.64267998770969, -3.470468960845686, -5.988304593016585],
 [-1.1395482530791987, 10.38043939479851, 8.942293880773892],
 [11.849190097810476, 3.9619039020134754, -3.5042652403575745],
 [12.486338369730682, 1.1890926335639567, 3.3414114047265886],
 [10.560484341142923, 6.663497227686959, 3.0796036364977506],
 [-7.142693776114026, 10.39088796770061, 0.7067799524073813],
 [9.975157439406106, 7.220341352256101, 2.5498298025599],
 [-6.479095316120745, 4.334022487716195, -9.047969749456282],
 [11.217719775747165, 2.2713995685169146, -3.36038895621125],
 [-9.826367661492673, 3.838625604063039, 4.927405933272353],
 [-8.764065977424158, 7.279564196119411, -0.5932967464418747],
 [-3.82001

In [25]:
print(len(angstrom_output))

29


In [27]:
model.eval()
with torch.no_grad():
    coord_context = torch.tensor([[33, 0, 0] for _ in range(75)]).unsqueeze(0)


    smiles_list = [
        ("Aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
        ("Caffeine", "Cn1cnc2c1c(=O)n(c(=O)n2C)C"),
        ("Ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ]

    for name, smi in smiles_list:
        fp = torch.tensor(smiles_fingerprint(smi),
                         dtype=torch.float32).unsqueeze(0).to(device)
        out, _ = model(coord_context, fp)
        print(f"{name}: {out.squeeze().tolist()}")

Aspirin: [16.111082077026367, -0.9513039588928223, 1.5987426042556763]
Caffeine: [17.573026657104492, -1.5455759763717651, 1.3449472188949585]
Ibuprofen: [17.53696632385254, 1.2823373079299927, 1.3318309783935547]


In [ ]:
# would be good to track n_trainable parameters